## Implicit Matrix Factorization (ALS)

As you have seen in the previous notebooks, we prepared temporal dataset splits and integrity checks, implemented the evaluation harness and introduced baselines and generated `leaderboard.csv`. # In this notebook, we introduce **Implicit Matrix Factorization (ALS)**, a core model for large-scale recommendation systems, especially when using implicit interactions such as clicks, watches, and purchases.



### What We're Achieving Here:

-  Train an Implicit ALS model on the train set  
-  Compute Recall@100 on validation and test splits  
-  Export learned item vectors for downstream tasks  

### Key Takeaways

- How to transform interaction logs into a sparse user–item matrix suitable for MF.  
- How implicit ALS works and why it is widely used for large-scale recommender systems.  
- How to evaluate retrieval performance using Recall@100, the standard for candidate generation.  
- How to export ALS latent factor embeddings for use in:
     - similarity models  
     - ANN search  
     - hybrid recommenders  
     - downstream ranking models  
- How this ALS model fits into the broader pipeline introduced in previous notebooks.  

### Agenda

1. Load dataset metadata and temporal splits  
2. Build the user–item interaction matrix
3. Train an implicit ALS model (64 latent factors)  
4. Implement an ALS-based recommendation function  
5. Evaluate Recall@100 on validation and test splits  
6. Export user and item embedding matrices for downstream use  
7. Save evaluation outputs

### Setup

In [ ]:
# we need to install implicit for the ALS model
! pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=10797470 sha256=51710c12f226e853fb586f301363403baf0866bd435e0a947f7a66cb529d0a7e
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [65]:
import pandas as pd
import numpy as np
import scipy.sparse as sps
import json
from pathlib import Path

### Load Metadata and Splits

In [66]:
base_path = "/content/"
meta = json.load(open(base_path + "split_metadata_20251103_1428.json"))
meta

{'timestamp': '20251103_1428',
 'total_interactions': 999917,
 'train_rows': 987837,
 'val_rows': 6040,
 'test_rows': 6040,
 'n_users': 6040,
 'n_items': 3503,
 'rating_threshold': 4.0,
 'val_k': 1,
 'test_k': 1}

### Load Train / Valid / Test Files

```
UserID | MovieID | Rating | Timestamp | Datetime | label
```

In [67]:
train = pd.read_csv(base_path + "train_20251103_1428.csv")
valid = pd.read_csv(base_path + "val_20251103_1428.csv")
test  = pd.read_csv(base_path + "test_20251103_1428.csv")

train.head()

,UserID,MovieID,Rating,Timestamp,Datetime,label
0,1,3186,4,978300019,2000-12-31 22:00:19+00:00,True
1,1,1270,5,978300055,2000-12-31 22:00:55+00:00,True
2,1,1721,4,978300055,2000-12-31 22:00:55+00:00,True
3,1,1022,5,978300055,2000-12-31 22:00:55+00:00,True
4,1,2340,3,978300103,2000-12-31 22:01:43+00:00,False


### Normalize Column Names

In [68]:
train = train.rename(columns={'UserID':'user_id','MovieID':'item_id'})
valid = valid.rename(columns={'UserID':'user_id','MovieID':'item_id'})
test  = test.rename(columns={'UserID':'user_id','MovieID':'item_id'})

train.shape, valid.shape, test.shape

((987837, 6), (6040, 8), (6040, 8))

### Build User/Item Index Maps

Next, we need to build the user-item maps. In order to build the user-item map the index order must match the matrix construction order.

In [69]:
user2idx = {}
item2idx = {}
idx2item = {}

row_idx = []
col_idx = []
data = []

user_list = []
item_list = []

In [70]:
for _, r in train.iterrows():
    u, i = r["user_id"], r["item_id"]

    if u not in user2idx:
        user2idx[u] = len(user2idx)
        user_list.append(u)

    if i not in item2idx:
        item2idx[i] = len(item2idx)
        item_list.append(i)

    row_idx.append(user2idx[u])
    col_idx.append(item2idx[i])
    data.append(1.0)

n_users = len(user2idx)
n_items = len(item2idx)

# Build matrix as USER × ITEM
mat = sps.csr_matrix((data, (row_idx, col_idx)), shape=(n_users, n_items))
print(f"User-Item matrix shape: {mat.shape}")  # Should be (6040, 3503)

User-Item matrix shape: (6040, 3503)


### Train Implicit ALS

ALS is trained on the **item-user matrix** (transpose of user–item). The implicit package performs Weighted Matrix Factorization using alternating least squares.

In [73]:
import implicit

als = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=20,
    use_gpu=False  # Explicitly disable GPU to avoid confusion
)

# Pass the matrix directly without transpose
# implicit expects User × Item format when using recommend()
als.fit(mat)

print(f"User factors shape: {als.user_factors.shape}")  # Should be (6040, 64)
print(f"Item factors shape: {als.item_factors.shape}") # Should be (3503, 64)

  0%|          | 0/20 [00:00<?, ?it/s]

User factors shape: (6040, 64)
Item factors shape: (3503, 64)


### Define Recommendation Function for ALS

We reuse the robust standardized recommender from the past exercise.

In [74]:
def normalize_recs(recs):
    """Normalize implicit recommend outputs into list of (item, score)."""
    if recs is None:
        return []
    # tuple of arrays (ids, scores)
    if isinstance(recs, tuple) and len(recs) == 2:
        return list(zip(recs[0], recs[1]))
    arr = np.asarray(recs)
    if arr.ndim == 2 and arr.shape[1] >= 2:
        return list(zip(arr[:,0].astype(int), arr[:,1].astype(float)))
    try:
        return [(int(x[0]), float(x[1])) for x in recs]
    except:
        return []

idx2item = {v:k for k,v in item2idx.items()}

In [75]:
def rec_als(user_id, K=100):
    if user_id not in user2idx:
        return []

    uidx = user2idx[user_id]

    # Ensure user index is valid
    if uidx >= mat.shape[0]:
        return []

    user_items = mat[uidx:uidx+1]

    try:
        raw = als.recommend(
            uidx,
            user_items,
            N=min(K * 2, n_items),  # Request more, but cap at total items
            filter_already_liked_items=False
        )
    except (IndexError, ValueError) as e:
        #print(f"Error for user {user_id}: {e}")
        return []

    parsed = normalize_recs(raw)

    # Map internal -> external ids WITH BOUNDS CHECKING
    result = []
    for i, score in parsed:
        # Only include valid indices within our item range
        if 0 <= i < n_items and i in idx2item:
            result.append(idx2item[i])
            if len(result) >= K:
                break

    return result

### Metric: Recall@100

We evaluate Recall@100 on both validation and test sets.

In [76]:
def recall_at_k(recs, gt):
    if len(gt) == 0:
        return 0.0
    hits = sum([1 for r in recs if r in gt])
    return hits / len(gt)

In [77]:
def eval_split(df, K=100):
    total = 0
    count = 0
    for user, group in df.groupby("user_id"):
        gt = set(group["item_id"])
        recs = rec_als(user, K)
        total += recall_at_k(recs, gt)
        count += 1
    return total / count

### Evaluate ALS — Recall@100

In [78]:
recall_val = eval_split(valid, K=100)
recall_test = eval_split(test,  K=100)

recall_val, recall_test

(0.2802980132450331, 0.2637417218543046)

### Export Item and User Vectors

These will be used by downstream models:
- Content-based similarity
- Approximate nearest neighbor search
- Two-tower architectures

In [79]:
item_vectors = als.item_factors
user_vectors = als.user_factors

In [80]:
np.save("als_item_vectors.npy", item_vectors)
np.save("als_user_vectors.npy", user_vectors)
print("Saved als_item_vectors.npy and als_user_vectors.npy")

Saved als_item_vectors.npy and als_user_vectors.npy


### Save ALS Evaluation Results

In [81]:
results = {
    "model": "als@factors=64",
    "recall@100": {
        "val": recall_val,
        "test": recall_test
    },
    "n_users": n_users,
    "n_items": n_items
}

json.dump(results, open("als_results.json", "w"), indent=2)
results

{'model': 'als@factors=64',
 'recall@100': {'val': 0.2802980132450331, 'test': 0.2637417218543046},
 'n_users': 6040,
 'n_items': 3503}

### Conclusion

 In this notebook, we trained a full implicit ALS model on the training interactions, evaluated the model’s ability to retrieve relevant items using Recall@100, and exported the learned user and item factor matrices. These embeddings now serve as core building blocks for downstream recommendation experiments, including similarity search, re-ranking models, or hybrid architectures. As in previous notebooks, all results were saved in standardised output files to integrate seamlessly with the rest of the pipeline.

In [60]:
# Diagnostic 1: Check if idx2item is complete
print("=== Mapping Diagnostics ===")
print(f"n_items from counter: {n_items}")
print(f"len(item2idx): {len(item2idx)}")
print(f"len(idx2item): {len(idx2item)}")
print(f"max(item2idx.values()): {max(item2idx.values())}")
print(f"min(item2idx.values()): {min(item2idx.values())}")
print(f"Matrix shape: {mat.shape}")
print(f"ALS item_factors shape: {als.item_factors.shape}")

# Diagnostic 2: Test a single recommendation
test_user = valid['user_id'].iloc[0]
print(f"\n=== Testing user {test_user} ===")
uidx = user2idx[test_user]
user_items = mat[uidx:uidx+1]
raw = als.recommend(uidx, user_items, N=10, filter_already_liked_items=False)
parsed = normalize_recs(raw)
print(f"Top 10 raw recommendations:")
for i, (item_idx, score) in enumerate(parsed[:10]):
    in_map = item_idx in idx2item
    ext_id = idx2item.get(item_idx, 'MISSING')
    print(f"  {i+1}. internal_id={item_idx}, score={score:.4f}, in_idx2item={in_map}, external_id={ext_id}")

# Diagnostic 3: Check how many recs are being filtered out
print("\n=== Filtering Analysis ===")
total_recs = 0
valid_recs = 0
invalid_recs = 0

for user in valid['user_id'].unique()[:100]:  # Check first 100 users
    if user not in user2idx:
        continue
    uidx = user2idx[user]
    user_items = mat[uidx:uidx+1]
    raw = als.recommend(uidx, user_items, N=100, filter_already_liked_items=False)
    parsed = normalize_recs(raw)
    total_recs += len(parsed)
    for i, _ in parsed:
        if i in idx2item:
            valid_recs += 1
        else:
            invalid_recs += 1

print(f"Total recommendations generated: {total_recs}")
print(f"Valid recommendations: {valid_recs} ({100*valid_recs/total_recs:.1f}%)")
print(f"Invalid recommendations filtered: {invalid_recs} ({100*invalid_recs/total_recs:.1f}%)")

=== Mapping Diagnostics ===
n_items from counter: 3503
len(item2idx): 3503
len(idx2item): 3503
max(item2idx.values()): 3502
min(item2idx.values()): 0
Matrix shape: (6040, 3503)
ALS item_factors shape: (6040, 64)

=== Testing user 1 ===
Top 10 raw recommendations:
  1. internal_id=2105, score=1.2042, in_idx2item=True, external_id=2819
  2. internal_id=530, score=1.1078, in_idx2item=True, external_id=778
  3. internal_id=4160, score=1.0749, in_idx2item=False, external_id=MISSING
  4. internal_id=1447, score=1.0586, in_idx2item=True, external_id=955
  5. internal_id=1604, score=1.0496, in_idx2item=True, external_id=2474
  6. internal_id=4084, score=0.9907, in_idx2item=False, external_id=MISSING
  7. internal_id=751, score=0.9884, in_idx2item=True, external_id=2105
  8. internal_id=4168, score=0.9859, in_idx2item=False, external_id=MISSING
  9. internal_id=3066, score=0.9794, in_idx2item=True, external_id=2810
  10. internal_id=2062, score=0.9756, in_idx2item=True, external_id=254

=== Fil